# Environment registry

> Register, list and create stable-marl environments by id.

In [ ]:
#| default_exp envs.registry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable

import gymnasium as gym
from gymnasium.envs.registration import registry as gym_registry
from omegaconf import DictConfig

from stable_marl.envs.base import MultiAgentEnv

In [ ]:
#| export
@dataclass(frozen=True)
class EnvSpec:
    "A registered environment: its id, the class that builds it and its default kwargs."
    id: str
    entry_point: Callable[..., MultiAgentEnv]
    kwargs: dict[str, Any] = field(default_factory=dict)
    family: str | None = None   # environment family, e.g. 'multigrid'


_REGISTRY: dict[str, EnvSpec] = {}


def register(
    id: str,
    entry_point: Callable[..., MultiAgentEnv],
    kwargs: dict[str, Any] | None = None,
    family: str | None = None) -> EnvSpec:
    """
    Register an environment under `id`, in stable-marl and in gymnasium
    (so ``gym.make(id)`` keeps working).
    """
    spec = EnvSpec(id, entry_point, dict(kwargs or {}), family)
    _REGISTRY[id] = spec
    if id not in gym_registry:
        gym.register(id=id, entry_point=entry_point, kwargs=spec.kwargs)
    return spec


def make(id: str, **kwargs) -> MultiAgentEnv:
    """
    Create the registered environment `id`, `kwargs` overriding its registered defaults.
    Unlike ``gym.make``, the env is returned unwrapped.
    """
    if id not in _REGISTRY:
        raise KeyError(f"Unknown environment {id!r}, registered ones are: {', '.join(list_envs())}")
    spec = _REGISTRY[id]
    return spec.entry_point(**{**spec.kwargs, **kwargs})


def list_envs(family: str | None = None) -> list[str]:
    "Ids of the registered environments, only those of `family` (e.g. 'multigrid') if given."
    return sorted(id for id, spec in _REGISTRY.items() if family is None or spec.family == family)

In [ ]:
#| export
def make_env(cfg: DictConfig):
    """
    Create an env from a Hydra config.
    Handles both raw envs and wrapped envs transparently.
    """
    if "_target_" in cfg:
        # Direct Hydra instantiation (raw env or custom wrapper)
        from hydra.utils import instantiate
        return instantiate(cfg)
    elif "gym_id" in cfg:
        # Gymnasium registry path
        kwargs = {k: v for k, v in cfg.items() if k != "gym_id"}
        return gym.make(cfg.gym_id, **kwargs)
    else:
        raise ValueError(f"Cannot create env from config: {cfg}")

In [ ]:
from stable_marl.envs.multigrid import EmptyEnv
register('Test-Empty-5x5-v0', EmptyEnv, {'size': 5}, family='test')
env = make('Test-Empty-5x5-v0', agents=2)
assert isinstance(env, MultiAgentEnv) and env.num_agents == 2 and env.width == 5
assert make('Test-Empty-5x5-v0', size=7).width == 7   # kwargs override the registered defaults
assert list_envs('test') == ['Test-Empty-5x5-v0'] and list_envs('no-such-family') == []
assert type(gym.make('Test-Empty-5x5-v0').unwrapped) is EmptyEnv   # also registered in gymnasium

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()